In [ ]:
import sys
sys.path.append('../')
from msBO import MultiStateBO
from msBO.objective import BPMvar_minimization
sys.path.append('../../machineIO/')
from machineIO import construct_machineIO, StatefulOracleEvaluator
from machineIO.preset import get_limits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from epics import caget, caget_many, caput_many
from phantasy import fetch_data
import datetime
from phantasy import ensure_set, fetch_data

In [ ]:
n_init = 16
n_each = 3
n_state = 2
n_iter = 4

In [ ]:
SCS = caget("ACS_DIAG:DEST:ACTIVE_ION_SOURCE")
ion = caget("FE_ISRC"+str(SCS)+":BEAM:ELMT_BOOK")
Q = caget("FE_ISRC"+str(SCS)+":BEAM:Q_BOOK")
A = caget("FE_ISRC"+str(SCS)+":BEAM:A_BOOK")
# AQ = caget("FE_ISRC2:BEAM:MOVRQ_BOOK")
AQ = A/Q
ion = str(A)+ion+str(Q)
print('SCS'+str(SCS), ion, 'A/Q=',AQ)

In [ ]:
now0 = datetime.datetime.now()
fname = now0.strftime('%Y%m%d_%H%M')+'['+ion+'][msBO]FS1'
fname

### construct machineIO

In [ ]:
mIO = construct_machineIO(ensure_set_timeout = 30, 
                          ensure_set_timewait_after_ramp = 0.3,
                          fetch_data_time_span = 2.0)

In [ ]:
mIO.__dict__

In [ ]:
ramping_time = 1
BO_comp_time = 5
dt = mIO._ensure_set_timewait_after_ramp + mIO._fetch_data_time_span + ramping_time + BO_comp_time
print(f'expected run time: {int((n_init*n_state + n_iter*n_state*n_each)*dt/60)} min')
print(f'expected number of data: {n_init*n_state + n_iter*n_state*n_each}')

### control knobs

In [ ]:
control_CSETs = [
    'MG5_ZHJ:PSC2_T50431921:I_CSET',
    'MG5_ZHJ:PSC1_T50431921:I_CSET',
    'MG5_ZHJ:PSC2_T50803624:I_CSET',
    'MG5_ZHJ:PSC1_T50803624:I_CSET',
#     'MG5_YQJ:PSC2_T51184268:I_CSET',
    'MG5_YQJ:PSC1_T51184268:I_CSET',
#     'MG5_YQJ:PSC2_T51959293:I_CSET',
    'MG5_YQJ:PSC1_T51959293:I_CSET',
    'MG5_YQJ:PSQ_T51233132:I_CSET',
    'MG5_YQJ:PSS_T51269742:I_CSET',
    'MG5_YBJ:PSC2_T52329580:I_CSET',
    'MG5_YBJ:PSC1_T52329580:I_CSET',
#     'MG5_YBJ:PSC2_T52649431:I_CSET',
#     'MG5_YBJ:PSC1_T52649431:I_CSET',
]
control_RDs = [pv.replace('CSET','RD') for pv in control_CSETs]


control_ref = np.array(caget_many(control_CSETs))
control_rd_ref = np.array(caget_many(control_RDs))
control_tols_ref = 3*np.abs(control_ref - control_rd_ref)

control_tols = []
control_min = []
control_max = []
i = 0
for v, PV in zip(control_ref,control_CSETs):
    if ':PSC' in PV:
        control_min.append(v - 2)
        control_max.append(v + 2)
        control_tols.append(max(0.1,control_tols_ref[i]))
    elif ':PSQ_' in PV:
        control_min.append(v*0.99)
        control_max.append(v*1.01)
        control_tols.append(max(0.1,control_tols_ref[i]))
    elif ':PSS_' in PV:
        control_min.append(v*0.3)
        control_max.append(v*1.7)
        control_tols.append(max(0.05,control_tols_ref[i]))
    else:
        raise ValueError(f'control min/max cannot be determined automatically for {PV}')
    i+=1
        
        
##== Manually set decision bounds and tolerance
# control_min = [ -5, -5, -5, -5]
# control_max = [  5,  5,  5,  5]
# control_tols = [0.2,0.2,0.2,0.2]

assert len(control_CSETs) == len(control_min) == len(control_max) == len(control_tols)
control_Lo_limit, control_Hi_limit = get_limits(control_CSETs)
control_min = np.clip(control_min, a_min = control_Lo_limit, a_max = None)
control_max = np.clip(control_max, a_min = None, a_max = control_Hi_limit)
assert np.all(control_max > control_min)

pd.DataFrame(np.array([control_ref,control_min,control_max,control_tols,control_Lo_limit,control_Hi_limit]).T,
             index=control_CSETs, 
             columns=['current value','decision min','decision max','tol','LoLim','HiLim'])

In [ ]:
control_couplings = {
    'MG5_YQJ:PSQ_T51233132:I_CSET':
        {
            'CSETs' :['MG5_YQJ:PSQ_T51330687:I_CSET','MG5_YQJ:PSQ_T51803028:I_CSET','MG5_YQJ:PSQ_T51911274:I_CSET'],
            'RDs'   :[],
            'tols'  :[],
            'coeffs':[],
        },
     'MG5_YQJ:PSS_T51269742:I_CSET':
        {
            'CSETs' :['MG5_YQJ:PSS_T51875223:I_CSET'],
            'RDs'   :[],
            'tols'  :[],
            'coeffs':[],
        },
}

for pv, val in control_couplings.items():
    x = caget(pv)
    y = np.array(caget_many(val['CSETs']))
    ipv = control_CSETs.index(pv)
    tol = control_tols[ipv]
    val['RDs'] = [pv.replace('CSET','RD') for pv in val['CSETs']]
    val['tols'] = [tol]*len(val['CSETs'])
    val['coeffs'] = y/x
    
control_couplings

### state definition

In [ ]:
state_CSETs  = ['MG5_YQJ:CSEL_T51098620:CTR_MTR.VAL']
state_RDs    = ['MG5_YQJ:CSEL_T51098620:CTR_MTR.RBV']
state_tols   = [0.1]*len(state_CSETs)
state_val0   = caget_many(state_CSETs)
state_key_vals  = {'26+':[  16.5],
                   '27+':[ -16.5]}
states = list(state_key_vals.keys())
n_states = len(states)
state_key_vals

### monitors

In [ ]:
# monitor_BPMs = ['MG5_YBJ:BPM_T52270079','MG5_YBJ:BPM_T52684819','MG5_YBJ:BPM_T53270242','MG5_YBJ:BPM_T53421115']
monitor_BPMs = ['MG5_YBJ:BPM_T52270079','MG5_YBJ:BPM_T52684819','MG5_YBJ:BPM_T53270242']
# monitor_BPMs = ['MG5_YBJ:BPM_T52270079','MG5_YBJ:BPM_T52684819']
monitor_RDs = []
for bpm in monitor_BPMs:
    monitor_RDs += [bpm+':XPOS_RD',bpm+':YPOS_RD',bpm+':MAG_RD']
#     monitor_RDs += [bpm+':YPOS_RD',bpm+':MAG_RD']

### Define Oracle 
# WARN!  This cell may change state!

In [ ]:
bpm_posRDs = [pv for pv in monitor_RDs if 'POS_RD' in pv]
bpm_magRDs = [pv for pv in monitor_RDs if 'MAG_RD' in pv]

BPM_MAGs_ref = {}
for state in list(state_key_vals.keys())[::-1]:
    input(f'set {state_CSETs} to {state_key_vals[state]}')
#     ret = ensure_set(state_CSETs,state_RDs,state_key_vals[state],state_tols,timeout=30)
    val, _ = fetch_data(bpm_magRDs,5)
    BPM_MAGs_ref[state] = val
BPM_MAGs_ref

In [ ]:
# BPM_MAGs_ref = {'27+': np.array([0.0105662 , 0.01204466, 0.01028153]),
#                 '26+': np.array([0.01445168, 0.01648626, 0.01410545])}

In [ ]:
# bpm_posRDs = [pv for pv in monitor_RDs if 'POS_RD' in pv]
# bpm_magRDs = [pv for pv in monitor_RDs if 'MAG_RD' in pv]

# BPM_MAGs_ref = {}
# for state, goal in state_key_vals.items():
#     ret = ensure_set(state_CSETs,state_RDs,goal,state_tols,timeout=30)
#     val, _ = fetch_data(bpm_magRDs,5)
#     BPM_MAGs_ref[state] = val
# BPM_MAGs_ref

In [ ]:
def BPM_MAG_obj(df,s):
    df['BPM:MAG_min_ratio'] = (df[bpm_magRDs].values/BPM_MAGs_ref[s][None,:]).min()
    return df

In [ ]:
oracle_key_names = {'x':control_RDs,
                    'y':bpm_posRDs + ['BPM:MAG_min_ratio']}
oracle_key_names

### check control couplings 

In [ ]:
oracleEvaluator = StatefulOracleEvaluator(
    mIO,
    control_CSETs= control_CSETs,
    control_RDs  = control_RDs,
    control_tols = control_tols,
    state_CSETs  = state_CSETs,
    state_RDs    = state_RDs,
    state_tols   = state_tols,
    state_key_vals = state_key_vals,
    oracle_key_names = oracle_key_names,
    monitor_PVs  = monitor_RDs,
    control_couplings = control_couplings,
    state_df_manipulators = [BPM_MAG_obj],
)

# Define Composite Object

In [ ]:
n_state = len(states)
n_task = len(bpm_posRDs)+1
composite_objective_function = BPMvar_minimization(S=n_state, J=n_task)

# Define msBO

In [ ]:
acq_type = 'EI'
fix_acq_state = False
local_optimization=False

msbo = MultiStateBO(
    states = states,
    tasks  = oracle_key_names['y'], 
    control_min = control_min,
    control_max = control_max,
    multistate_oracle_evaluator = oracleEvaluator,
    composite_objective_function = composite_objective_function,
    local_bound_size = 0.2*(np.asarray(control_max) - np.asarray(control_min)),
    model_train_epochs     = 200,
    model_warmstart_epochs = 50,
    )

# run msBO

In [ ]:
msbo.init(n_init=n_init, local_optimization=local_optimization)
for i_iter in range(n_iter):
    if i_iter%2==0:
        states_order = states[::-1]
    else:
        states_order = states
    for s in states_order:
        print(i_iter, s)
        msbo.step_batch(s=s, q=n_each,
                        local_optimization=local_optimization, 
                        acq_type=acq_type, fix_acq_state=fix_acq_state)
    fig, axes = msbo.plot_state_predictions_history()
    plt.show()

In [ ]:
now1 = datetime.datetime.now()
timedelta = now1-now0
timedelta

### visualize optimization result

In [ ]:
fig,ax, virtualObjMean = msbo.plot_composite_objective()
ax[0].set_ylim(0.9,1)

In [ ]:
fig, axes = msbo.plot_state_predictions_history()

### set to best solution

In [ ]:
from epics import caput_many, caget_many

x_best = msbo.history['predictions'][-1]['x_best']
caput_many(control_CSETs,x_best)

### restore states

In [ ]:
caput_many(state_CSETs,state_val0)

### restore control if optimization result is not good

In [ ]:
# caput_many(control_CSETs,control_ref)